**Objetivo:** Construir un chatbot que responda preguntas en base a un dataset de ventas usando RAG (Retrieval-Augmented Generation) con el modelo Mistral via API.
## ¿Qué es cada concepto?
- **Corpus:** El conjunto de datos/documentos que le damos al modelo como base de conocimiento. En este caso, nuestro dataset de ventas.
- **RAG (Retrieval-Augmented Generation):** Técnica que combina búsqueda de información relevante + generación de texto. En vez de 'entrenar' el modelo, le pasamos contexto relevante en cada pregunta.
- **Agentes:** Componentes que toman decisiones y ejecutan acciones (en este caso, el agente decide qué parte del corpus buscar).
- **Mistral:** Modelo de lenguaje de código abierto (francés). Usamos su API, similar a como funciona ChatGPT pero con Mistral.

## PASO 1 — Instalación de librerías

In [1]:
# Instalamos las librerías necesarias
# - pandas: para manejar el dataset CSV
# - mistralai: cliente oficial de la API de Mistral
# - langchain + langchain-community: framework para construir pipelines RAG
# - faiss-cpu: motor de búsqueda vectorial (para el RAG)
# - sentence-transformers: para convertir texto en vectores numéricos

!pip install -q pandas mistralai langchain langchain-community faiss-cpu sentence-transformers

## PASO 2 — Configurar la API Key de Mistral

In [2]:
from google.colab import userdata
import os

# Leemos la API key desde Secrets de Colab
MISTRAL_API_KEY = userdata.get('ventas')

# La guardamos también como variable de entorno (buena práctica)
os.environ['ventas'] = MISTRAL_API_KEY

print('✅ API Key cargada correctamente')

✅ API Key cargada correctamente


## PASO 3 — Carga y exploración del dataset original

In [3]:
import pandas as pd

# Cargamos el dataset original (sin normalizar)
# encoding latin-1 porque tiene caracteres especiales europeos
df_raw = pd.read_csv('sales_data_sample.csv', encoding='latin-1')

print('=== DIMENSIONES DEL DATASET ===')
print(f'Filas: {df_raw.shape[0]}')
print(f'Columnas: {df_raw.shape[1]}')

print('\n=== PRIMERAS FILAS ===')
df_raw.head()

=== DIMENSIONES DEL DATASET ===
Filas: 2823
Columnas: 25

=== PRIMERAS FILAS ===


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [4]:
print('=== TIPOS DE DATOS (variables) ===')
print(df_raw.dtypes)

print('\n=== VALORES NULOS POR COLUMNA ===')
print(df_raw.isnull().sum())

=== TIPOS DE DATOS (variables) ===
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

=== VALORES NULOS POR COLUMNA ===
ORDERNUMBER            0
QUANTITYORDERED        0
PRICEEACH              0
ORDERLINENUMBER        0
SALES                  0
ORDERDATE              0
STATUS                 0
QTR_ID                 0
MONTH_ID        

In [5]:
print('=== ESTADÍSTICAS DESCRIPTIVAS (variables numéricas) ===')
df_raw.describe()

=== ESTADÍSTICAS DESCRIPTIVAS (variables numéricas) ===


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


In [6]:
print('=== VARIABLES CATEGÓRICAS ÚNICAS ===')
categoricas = ['STATUS', 'PRODUCTLINE', 'DEALSIZE', 'TERRITORY', 'COUNTRY']
for col in categoricas:
    print(f'\n{col}: {df_raw[col].unique()}')

=== VARIABLES CATEGÓRICAS ÚNICAS ===

STATUS: ['Shipped' 'Disputed' 'In Process' 'Cancelled' 'On Hold' 'Resolved']

PRODUCTLINE: ['Motorcycles' 'Classic Cars' 'Trucks and Buses' 'Vintage Cars' 'Planes'
 'Ships' 'Trains']

DEALSIZE: ['Small' 'Medium' 'Large']

TERRITORY: [nan 'EMEA' 'APAC' 'Japan']

COUNTRY: ['USA' 'France' 'Norway' 'Australia' 'Finland' 'Austria' 'UK' 'Spain'
 'Sweden' 'Singapore' 'Canada' 'Japan' 'Italy' 'Denmark' 'Belgium'
 'Philippines' 'Germany' 'Switzerland' 'Ireland']


## PASO 4 — Normalización del dataset

**¿Por qué normalizar?** El dataset tiene varios problemas que afectan la calidad del análisis:
- Fechas en formato texto (no reconocidas como fecha)
- Valores nulos en columnas como ADDRESSLINE2, STATE, POSTALCODE
- Caracteres especiales corruptos (encoding incorrecto)
- Columnas redundantes o irrelevantes para el chatbot
- Tipos de datos incorrectos

In [7]:
import pandas as pd

# Recargamos con encoding correcto
df = pd.read_csv('sales_data_sample.csv', encoding='latin-1')

# --- 1. Normalizar fecha ---
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], format='%m/%d/%Y %H:%M')
print('✅ Fechas convertidas')

# --- 2. Rellenar valores nulos ---
# Columnas de dirección: rellenamos con 'N/A'
df['ADDRESSLINE2'] = df['ADDRESSLINE2'].fillna('N/A')
df['STATE'] = df['STATE'].fillna('N/A')
df['POSTALCODE'] = df['POSTALCODE'].fillna('N/A')
df['TERRITORY'] = df['TERRITORY'].fillna('N/A')
print('✅ Valores nulos rellenados')

# --- 3. Normalizar tipos numéricos ---
df['QUANTITYORDERED'] = df['QUANTITYORDERED'].astype(int)
df['PRICEEACH'] = df['PRICEEACH'].astype(float)
df['SALES'] = df['SALES'].astype(float)
df['MSRP'] = df['MSRP'].astype(int)
print('✅ Tipos numéricos normalizados')

# --- 4. Normalizar texto: mayúsculas consistentes ---
df['STATUS'] = df['STATUS'].str.strip()
df['PRODUCTLINE'] = df['PRODUCTLINE'].str.strip()
df['DEALSIZE'] = df['DEALSIZE'].str.strip()
df['COUNTRY'] = df['COUNTRY'].str.strip()
print('✅ Texto normalizado')

# --- 5. Eliminar duplicados ---
antes = len(df)
df = df.drop_duplicates()
print(f'✅ Duplicados eliminados: {antes - len(df)} filas removidas')

# --- 6. Guardar dataset normalizado con nuevo nombre ---
df.to_csv('sales_data_normalizado.csv', index=False, encoding='utf-8')
print('\n✅ Dataset normalizado guardado como: sales_data_normalizado.csv')

print(f'\n=== DIMENSIONES FINALES ===')
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
df.head()

✅ Fechas convertidas
✅ Valores nulos rellenados
✅ Tipos numéricos normalizados
✅ Texto normalizado
✅ Duplicados eliminados: 0 filas removidas

✅ Dataset normalizado guardado como: sales_data_normalizado.csv

=== DIMENSIONES FINALES ===
Filas: 2823
Columnas: 25


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,N/A,NYC,NY,10022,USA,N/A,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,N/A,Reims,N/A,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,N/A,Paris,N/A,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,...,78934 Hillside Dr.,N/A,Pasadena,CA,90003,USA,N/A,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,...,7734 Strong St.,N/A,San Francisco,CA,N/A,USA,N/A,Brown,Julie,Medium


In [8]:
# Verificación final: no deben quedar nulos
print('=== VERIFICACIÓN: NULOS RESTANTES ===')
print(df.isnull().sum())

print('\n=== TIPOS DE DATOS FINALES ===')
print(df.dtypes)

=== VERIFICACIÓN: NULOS RESTANTES ===
ORDERNUMBER         0
QUANTITYORDERED     0
PRICEEACH           0
ORDERLINENUMBER     0
SALES               0
ORDERDATE           0
STATUS              0
QTR_ID              0
MONTH_ID            0
YEAR_ID             0
PRODUCTLINE         0
MSRP                0
PRODUCTCODE         0
CUSTOMERNAME        0
PHONE               0
ADDRESSLINE1        0
ADDRESSLINE2        0
CITY                0
STATE               0
POSTALCODE          0
COUNTRY             0
TERRITORY           0
CONTACTLASTNAME     0
CONTACTFIRSTNAME    0
DEALSIZE            0
dtype: int64

=== TIPOS DE DATOS FINALES ===
ORDERNUMBER                  int64
QUANTITYORDERED              int64
PRICEEACH                  float64
ORDERLINENUMBER              int64
SALES                      float64
ORDERDATE           datetime64[ns]
STATUS                      object
QTR_ID                       int64
MONTH_ID                     int64
YEAR_ID                      int64
PRODUCTLINE      

## PASO 5 — Construcción del Corpus para RAG

El **corpus** es la representación textual de los datos que el chatbot usará como fuente de conocimiento.
Para RAG, convertimos cada fila del dataset en un texto descriptivo ('documento').

In [9]:
# Cargamos el dataset normalizado
df = pd.read_csv('sales_data_normalizado.csv', encoding='utf-8')

def fila_a_texto(row):
    """Convierte una fila del dataset en un texto descriptivo para el corpus."""
    return (
        f"Orden {row['ORDERNUMBER']}: Se vendieron {row['QUANTITYORDERED']} unidades "
        f"del producto {row['PRODUCTCODE']} (línea: {row['PRODUCTLINE']}) "
        f"a ${row['PRICEEACH']} cada una, con ventas totales de ${row['SALES']}. "
        f"Estado del pedido: {row['STATUS']}. "
        f"Cliente: {row['CUSTOMERNAME']} de {row['CITY']}, {row['COUNTRY']}. "
        f"Tamaño del trato: {row['DEALSIZE']}. "
        f"Fecha: {row['ORDERDATE']}. "
        f"Año: {row['YEAR_ID']}, Trimestre: {row['QTR_ID']}, Mes: {row['MONTH_ID']}."
    )

# Aplicamos la función a cada fila
corpus = df.apply(fila_a_texto, axis=1).tolist()

print(f'✅ Corpus generado: {len(corpus)} documentos')
print('\n=== EJEMPLO DE DOCUMENTO DEL CORPUS ===')
print(corpus[0])

✅ Corpus generado: 2823 documentos

=== EJEMPLO DE DOCUMENTO DEL CORPUS ===
Orden 10107: Se vendieron 30 unidades del producto S10_1678 (línea: Motorcycles) a $95.7 cada una, con ventas totales de $2871.0. Estado del pedido: Shipped. Cliente: Land of Toys Inc. de NYC, USA. Tamaño del trato: Small. Fecha: 2003-02-24. Año: 2003, Trimestre: 1, Mes: 2.


## PASO 6 — Vectorización del Corpus (Base de datos vectorial con FAISS)

Para que el RAG funcione, necesitamos convertir cada documento de texto en un **vector numérico** (embedding).  
Luego FAISS nos permite buscar los documentos más similares a una pregunta en milisegundos.

In [10]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Cargando modelo de embeddings...')
# Usamos un modelo multilingüe para soporte de español
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Generando embeddings del corpus (puede tardar unos minutos)...')
embeddings = modelo_embeddings.encode(corpus, show_progress_bar=True)

# Convertimos a float32 (requerido por FAISS)
embeddings = np.array(embeddings).astype('float32')

# Creamos el índice FAISS
dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatL2(dimension)
indice_faiss.add(embeddings)

print(f'\n✅ Índice FAISS creado con {indice_faiss.ntotal} vectores')
print(f'Dimensión de cada vector: {dimension}')

Cargando modelo de embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando embeddings del corpus (puede tardar unos minutos)...


Batches:   0%|          | 0/89 [00:00<?, ?it/s]


✅ Índice FAISS creado con 2823 vectores
Dimensión de cada vector: 384


## PASO 7 — Función de Recuperación (Retrieval)

Esta función toma una pregunta, la vectoriza, y busca los documentos más relevantes del corpus.

In [11]:
def recuperar_contexto(pregunta, top_k=5):
    """
    Dado una pregunta, retorna los top_k documentos más relevantes del corpus.

    Args:
        pregunta: texto de la pregunta del usuario
        top_k: cuántos documentos recuperar (más = más contexto, pero más tokens)

    Returns:
        Texto con los documentos relevantes concatenados
    """
    # Vectorizamos la pregunta
    vector_pregunta = modelo_embeddings.encode([pregunta]).astype('float32')

    # Buscamos los más similares en FAISS
    distancias, indices = indice_faiss.search(vector_pregunta, top_k)

    # Recuperamos los documentos correspondientes
    documentos_relevantes = [corpus[i] for i in indices[0]]

    return '\n'.join(documentos_relevantes)

# Prueba de la función
print('=== PRUEBA DE RECUPERACIÓN ===')
test = recuperar_contexto('ventas de motocicletas en Francia')
print(test[:800], '...')

=== PRUEBA DE RECUPERACIÓN ===
Orden 10134: Se vendieron 30 unidades del producto S18_2625 (línea: Motorcycles) a $61.78 cada una, con ventas totales de $1853.4. Estado del pedido: Shipped. Cliente: Lyon Souveniers de Paris, France. Tamaño del trato: Small. Fecha: 2003-07-01. Año: 2003, Trimestre: 3, Mes: 7.
Orden 10134: Se vendieron 35 unidades del producto S24_1578 (línea: Motorcycles) a $93.54 cada una, con ventas totales de $3273.9. Estado del pedido: Shipped. Cliente: Lyon Souveniers de Paris, France. Tamaño del trato: Medium. Fecha: 2003-07-01. Año: 2003, Trimestre: 3, Mes: 7.
Orden 10134: Se vendieron 41 unidades del producto S10_1678 (línea: Motorcycles) a $94.74 cada una, con ventas totales de $3884.34. Estado del pedido: Shipped. Cliente: Lyon Souveniers de Paris, France. Tamaño del trato: Medium. Fecha: 2003- ...


## PASO 8 — Selección del modelo Mistral

**¿Qué modelo usar?** Considerando el contexto:
- Dataset en inglés, preguntas posiblemente en español
- Tarea de QA (preguntas y respuestas) sobre datos estructurados
- API con límite de uso (probablemente plan gratuito)

**Decisión: `mistral-small-latest`** (anteriormente llamado `mistral-7b-instruct`)
- Rápido y eficiente para QA sobre datos
- Soporte multilingüe (entiende español)
- Menor costo de tokens vs modelos más grandes
- Ideal cuando el contexto ya viene filtrado por RAG


In [17]:
from mistralai.client import Mistral

# Inicializamos el cliente de Mistral
cliente_mistral = Mistral(api_key=MISTRAL_API_KEY)

# Modelo seleccionado: mistral-small-latest
# Justificación: eficiente para RAG + QA, soporte multilingüe, menor costo
MODELO_MISTRAL = 'mistral-small-latest'

print(f'✅ Cliente Mistral inicializado con modelo: {MODELO_MISTRAL}')

✅ Cliente Mistral inicializado con modelo: open-mistral-7b


## PASO 9 — Construcción del Chatbot RAG

Aquí unimos todo: recuperación de contexto + generación con Mistral.


In [18]:
def chatbot_ventas(pregunta, top_k=5):
    """
    Chatbot RAG que responde preguntas sobre el dataset de ventas.

    Pipeline:
    1. Recupera contexto relevante del corpus (RAG)
    2. Construye un prompt con el contexto + pregunta
    3. Envía a Mistral y retorna la respuesta
    """
    print(f'🔍 Buscando información relevante...')
    contexto = recuperar_contexto(pregunta, top_k=top_k)

    # Construimos el prompt con instrucciones claras para el modelo
    prompt_sistema = """Eres un asistente de análisis de ventas experto.
Responde preguntas basándote ÚNICAMENTE en el contexto de datos de ventas que se te proporciona.
Si la información no está en el contexto, dilo claramente.
Responde en el mismo idioma en que te hacen la pregunta.
Sé conciso, preciso y útil. Si hay números, menciónalos."""

    prompt_usuario = f"""Contexto de datos de ventas:
{contexto}

Pregunta: {pregunta}

Responde basándote en los datos del contexto:"""

    # Llamamos a la API de Mistral
    respuesta = cliente_mistral.chat.complete(
        model=MODELO_MISTRAL,
        messages=[
            {'role': 'system', 'content': prompt_sistema},
            {'role': 'user', 'content': prompt_usuario}
        ],
        max_tokens=500,
        temperature=0.3  # Baja temperatura = respuestas más precisas/consistentes
    )

    return respuesta.choices[0].message.content


print('✅ Función chatbot_ventas() lista')

✅ Función chatbot_ventas() lista


## PASO 10 — Pruebas del Chatbot

In [19]:
# Prueba 1: pregunta en español sobre línea de producto
pregunta1 = '¿Cuáles son las ventas de motocicletas más altas registradas?'
print(f'❓ {pregunta1}')
print(f'💬 {chatbot_ventas(pregunta1)}')
print('---')

❓ ¿Cuáles son las ventas de motocicletas más altas registradas?
🔍 Buscando información relevante...
💬 Basándome en el contexto proporcionado, las **ventas totales más altas registradas** en motocicletas son:

**$4,351.23** (Orden 10201) del producto **S24_1578** (39 unidades a $100.0 cada una), vendido a **Mini Wheels Co.** en **2003-12-01**.
---


In [20]:
# Prueba 2: pregunta sobre clientes
pregunta2 = '¿Qué clientes compraron Classic Cars y de qué país son?'
print(f'❓ {pregunta2}')
print(f'💬 {chatbot_ventas(pregunta2)}')
print('---')

❓ ¿Qué clientes compraron Classic Cars y de qué país son?
🔍 Buscando información relevante...
💬 Basándome en el contexto proporcionado, los clientes que compraron productos de la línea **Classic Cars** son:

1. **AV Stores, Co.** – **Reino Unido (UK)**
2. **Men 'R' US Retailers, Ltd.** – **Estados Unidos (USA)**
3. **Toms Spezialitten, Ltd.** – **Alemania (Germany)**
4. **Blauer See Auto, Co.** – **Alemania (Germany)**
---


In [21]:
# Prueba 3: pregunta analítica
pregunta3 = '¿Cuál es el estado más común de los pedidos en los datos?'
print(f'❓ {pregunta3}')
print(f'💬 {chatbot_ventas(pregunta3)}')
print('---')

❓ ¿Cuál es el estado más común de los pedidos en los datos?
🔍 Buscando información relevante...
💬 En los datos proporcionados, **todos los pedidos tienen el estado "Shipped"**.

No hay ningún pedido con otro estado diferente.
---


In [22]:
# Prueba 4: pregunta en inglés
pregunta4 = 'What is the average price per unit for large deals?'
print(f'❓ {pregunta4}')
print(f'💬 {chatbot_ventas(pregunta4)}')
print('---')

❓ What is the average price per unit for large deals?
🔍 Buscando información relevante...
💬 En el contexto proporcionado **no hay ningún pedido clasificado como *Large*** (grande). Los tamaños de trato registrados son **Medium** y **Small**.

Por lo tanto, **no es posible calcular el precio promedio por unidad para tratos grandes** con los datos disponibles.
---
